# Walk-forward lab -- IS/OOS splits & progressive tuning (FWPT)

**Evaluating stability** (findings Part 4). Two questions:
1. *IS/OOS* -- does a threshold that wins on the first 70% still win on
   the unseen last 30%? A value that only wins in-sample is noise.
2. *Forward-walk progressive tuning* -- expanding-window folds pick the
   best value on everything-before-the-boundary, then bank the winner's
   trades on the next unseen segment. The **pooled OOS line is the honest
   answer**; whole-history numbers overfit.

This is the promotion gate: a change earns its keep only by beating
baseline on pooled OOS with a *stable* per-fold winner.

In [ ]:
from pathlib import Path
import subprocess, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / 'config.json').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from rhengine import viz
viz.use_style()
CFG = json.load(open(ROOT / 'config.json'))
RESULTS = ROOT / 'tests' / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)

# Full-market event-loop passes are slow; cap the universe for a live
# notebook run and raise it (or set 0) when you want the real number.
MAX_TICKERS  = 150
LOOKBACK     = 400
COST_BPS     = 5
REFRESH      = False   # True -> recompute cached CSVs even if present
print('root:', ROOT, '| data_source:', CFG.get('data_source'))

In [ ]:
STRATEGY = 'bollinger'
PARAMS   = ['volMult', 'adxMax']   # thresholds to sweep (from strategies.json)
SPLIT    = 0.7
FOLDS    = 4
MIN_SCORE = 3

from rhengine.optimize import sweep_oat, walk_forward
from rhengine.indicators import spec_defaults
from rhengine.data_local import load_bars_any
bars_map, spy_bars = load_bars_any(CFG, lookback_days=LOOKBACK, verbose=False)
if MAX_TICKERS: bars_map = dict(list(bars_map.items())[:MAX_TICKERS])
defaults = spec_defaults(STRATEGY)
print(len(bars_map), 'tickers | defaults:', {p: defaults.get(p) for p in PARAMS})

## 1 -- One-at-a-time IS/OOS sweep

For each candidate value: in-sample stats (train) beside out-of-sample
(test). `sweep_oat` now returns the rows so we can chart them.

In [ ]:
rows = sweep_oat(CFG, bars_map, spy_bars, STRATEGY, PARAMS, MIN_SCORE,
                 COST_BPS, SPLIT, min_trades=15, max_values=9)
df = pd.DataFrame([{ 'param': r['param'], 'value': r['value'],
        'is_avgR': (r['is'] or {}).get('avg_r', np.nan),
        'oos_avgR': (r['oos'] or {}).get('avg_r', np.nan),
        'is_n': (r['is'] or {}).get('n', 0), 'oos_n': (r['oos'] or {}).get('n', 0)}
    for r in rows])
df

In [ ]:
fig, axes = plt.subplots(1, len(PARAMS), figsize=(6.2*len(PARAMS), 4.2), squeeze=False)
for ax, p in zip(axes.flat, PARAMS):
    d = df[df.param == p]
    viz.sweep_lines(d['value'].tolist(), d['is_avgR'].tolist(),
        d['oos_avgR'].tolist(), p, default=defaults.get(p),
        title=f'{STRATEGY}: {p} IS vs OOS', ax=ax)
plt.tight_layout(); plt.show()

## 2 -- Forward-walk progressive tuning

Expanding-window folds. Each fold tunes on the past, tests on the next
unseen block. Watch two things: the **pooled OOS** stat, and whether the
per-fold winner is *stable* (same value each fold) or a coin flip.

In [ ]:
pooled, fold_log = walk_forward(CFG, bars_map, spy_bars, STRATEGY, PARAMS,
        MIN_SCORE, COST_BPS, FOLDS, min_trades=15, max_values=9)
flog = pd.DataFrame([{ 'fold': f['fold'], 'chosen': f['chosen'],
        'train_avgR': (f['train'] or {}).get('avg_r', np.nan),
        'test_avgR': (f['test'] or {}).get('avg_r', np.nan),
        'test_n': (f['test'] or {}).get('n', 0)} for f in fold_log])
flog

In [ ]:
fig, ax = viz.paired_bars(
    [f"f{r.fold}\n{r.chosen[:16]}" for r in flog.itertuples()],
    flog['train_avgR'].tolist(), flog['test_avgR'].tolist(),
    label_a='train (IS)', label_b='test (OOS)', ylabel='avg R',
    title=f'{STRATEGY} FWPT -- per-fold train vs unseen-test avg R')
plt.show()
chosen = set(flog['chosen'])
print(f'distinct winners across folds: {len(chosen)} ->',
      'STABLE' if len(chosen) <= 2 else 'UNSTABLE (the best value is noise)')

## 3 -- Pooled out-of-sample verdict

The single number that decides promotion. Compare to the defaults
baseline (run this notebook with `PARAMS=[]`-equivalent by reading the
`(defaults)` fold rows).

In [ ]:
from rhengine.backtest import _stats
st = _stats(pooled)
if st:
    print(f"POOLED OOS: n={st['n']} win={st['win_rate']*100:.1f}% "
          f"avgR={st['avg_r']:+.3f} PF={st['profit_factor']:.2f} t={st['t_stat']:+.1f}")
    viz.dist_hist([t['r_multiple'] for t in pooled], bins=50,
        title=f'{STRATEGY} pooled-OOS R-multiple distribution',
        xlabel='R', x_fmt='{:+.1f}', ref=st['avg_r'], ref_label=f"mean {st['avg_r']:+.3f}")
    plt.show()
else:
    print('no pooled trades')

---
**Promotion bar:** pooled OOS avg R beats the defaults baseline, the
per-fold winner is stable, and t-stat >= 2. Anything that wins IS but
not pooled-OOS stays off. Encode the winner in `strategies.json`
(default/min/max/step) and `config.json`, then re-run the ablation and
Monte Carlo labs.